In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 04. Full Pipeline Demo\n",
    "Complete Monte Carlo simulation with DFMGAN + PEMC"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "import sys\n",
    "import os\n",
    "sys.path.append(os.path.abspath('..'))\n",
    "\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "from pprint import pprint\n",
    "\n",
    "from src.pipeline import MonteCarloPipeline\n",
    "from src.explainer.llama_wrapper import LlamaExplainer\n",
    "\n",
    "%matplotlib inline\n",
    "\n",
    "print(\"[INFO] Environment ready\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Initialize pipeline safely\n",
    "try:\n",
    "    pipeline = MonteCarloPipeline(\n",
    "        n_assets=3,\n",
    "        n_simulations=5000,\n",
    "        filter_top_k=0.01\n",
    "    )\n",
    "    \n",
    "    pipeline.load_models()\n",
    "    print(\"[SUCCESS] Pipeline ready\")\n",
    "except Exception as e:\n",
    "    print(f\"[ERROR] Pipeline initialization failed: {e}\")\n",
    "    raise"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Run simulation\n",
    "tickers = ['AAPL', 'MSFT', 'GOOGL']\n",
    "\n",
    "try:\n",
    "    results = pipeline.run_simulation(tickers, period='2y')\n",
    "\n",
    "    metadata = results.get('metadata', {})\n",
    "    print(f\"Simulation completed in {metadata.get('computation_time', 0):.2f}s\")\n",
    "    print(f\"Generated {metadata.get('n_simulations', 0)} paths\")\n",
    "    print(f\"Filtered to {metadata.get('filtered_paths', 0)} most realistic\")\n",
    "except Exception as e:\n",
    "    print(f\"[ERROR] Simulation failed: {e}\")\n",
    "    raise"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Display results safely\n",
    "print(\"\\nExpected Prices:\")\n",
    "\n",
    "expected_prices = results.get('expected_prices', {})\n",
    "confidence_intervals = results.get('confidence_intervals', {})\n",
    "\n",
    "for ticker in tickers:\n",
    "    if ticker in expected_prices:\n",
    "        exp = expected_prices[ticker]\n",
    "        ci_low, ci_high = confidence_intervals.get(ticker, (0, 0))\n",
    "        print(f\"{ticker}: ${exp:.2f} [${ci_low:.2f}, ${ci_high:.2f}]\")\n",
    "    else:\n",
    "        print(f\"{ticker}: No data\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Risk metrics (safe access)\n",
    "print(\"\\nRisk Metrics:\")\n",
    "\n",
    "risk_metrics = results.get('risk_metrics', {})\n",
    "\n",
    "for ticker, metrics in risk_metrics.items():\n",
    "    print(f\"\\n{ticker}:\")\n",
    "    print(f\"  VaR (95%): {metrics.get('var_95', 0)*100:.1f}%\")\n",
    "    print(f\"  CVaR (95%): {metrics.get('cvar_95', 0)*100:.1f}%\")\n",
    "    print(f\"  Sharpe: {metrics.get('sharpe', 0):.2f}\")\n",
    "    print(f\"  Expected Return: {metrics.get('expected_return', 0)*100:.1f}%\")\n",
    "    print(f\"  Volatility: {metrics.get('volatility', 0)*100:.1f}%\")\n",
    "    print(f\"  Max Drawdown: {metrics.get('max_drawdown', 0)*100:.1f}%\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Plot sample paths (robust)\n",
    "if 'path_sample' in results:\n",
    "    paths = np.array(results['path_sample'])\n",
    "\n",
    "    if paths.ndim == 3:\n",
    "        fig, axes = plt.subplots(1, min(3, paths.shape[2]), figsize=(15, 4))\n",
    "\n",
    "        if paths.shape[2] == 1:\n",
    "            axes = [axes]\n",
    "\n",
    "        for i, ticker in enumerate(tickers[:paths.shape[2]]):\n",
    "            ax = axes[i]\n",
    "\n",
    "            for j in range(min(10, paths.shape[0])):\n",
    "                ax.plot(paths[j, :, i], alpha=0.3, linewidth=0.7)\n",
    "\n",
    "            mean_path = np.mean(paths[:, :, i], axis=0)\n",
    "            ax.plot(mean_path, 'k--', linewidth=2, label='Mean')\n",
    "\n",
    "            ax.set_title(f'{ticker} Sample Paths')\n",
    "            ax.set_xlabel('Days')\n",
    "            ax.set_ylabel('Price')\n",
    "            ax.legend()\n",
    "            ax.grid(True, alpha=0.3)\n",
    "\n",
    "        plt.tight_layout()\n",
    "        plt.show()\n",
    "    else:\n",
    "        print(\"[WARNING] Unexpected path shape\")\n",
    "else:\n",
    "    print(\"[INFO] No path samples available\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Variance reduction\n",
    "vr = results.get('variance_reduction', 0)\n",
    "print(f\"\\nVariance Reduction Achieved: {vr*100:.2f}%\")\n",
    "print(\"(Typical range: 30-55% if model is strong)\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# LLM Explanation (safe + optional)\n",
    "model_path = \"../models/llama-3.1-1b\"\n",
    "\n",
    "if os.path.exists(model_path):\n",
    "    try:\n",
    "        print(\"\\n[INFO] Generating LLaMA explanation...\")\n",
    "\n",
    "        explainer = LlamaExplainer(model_path)\n",
    "\n",
    "        explanation = explainer.explain_simulation_results(\n",
    "            tickers=tickers,\n",
    "            expected_prices=expected_prices,\n",
    "            confidence_intervals=confidence_intervals,\n",
    "            risk_metrics=risk_metrics,\n",
    "            variance_reduction=vr\n",
    "        )\n",
    "\n",
    "        print(\"\\n\" + \"=\"*60)\n",
    "        print(\"LLaMA EXPLANATION\")\n",
    "        print(\"=\"*60)\n",
    "        print(explanation)\n",
    "\n",
    "    except Exception as e:\n",
    "        print(f\"[WARNING] LLaMA failed: {e}\")\n",
    "else:\n",
    "    print(\"\\n[INFO] LLaMA model not found. Skipping explanation.\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Save results safely\n",
    "import json\n",
    "\n",
    "os.makedirs('../data', exist_ok=True)\n",
    "\n",
    "def convert_to_serializable(obj):\n",
    "    if isinstance(obj, np.integer):\n",
    "        return int(obj)\n",
    "    elif isinstance(obj, np.floating):\n",
    "        return float(obj)\n",
    "    elif isinstance(obj, np.ndarray):\n",
    "        return obj.tolist()\n",
    "    return obj\n",
    "\n",
    "try:\n",
    "    serializable_results = json.loads(\n",
    "        json.dumps(results, default=convert_to_serializable)\n",
    "    )\n",
    "\n",
    "    with open('../data/results.json', 'w') as f:\n",
    "        json.dump(serializable_results, f, indent=2)\n",
    "\n",
    "    print(\"\\n[SUCCESS] Results saved to ../data/results.json\")\n",
    "\n",
    "except Exception as e:\n",
    "    print(f\"[ERROR] Saving failed: {e}\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}
